# Notebook 0: Getting Started

This notebook has one job: get your computer connected to the Domain API and pull back real data.

By the end, you will have run a successful query and seen live property market statistics for a Melbourne suburb. Everything in Notebooks 1–4 builds on exactly what you do here.

**No prior programming experience is assumed** beyond being able to open a terminal and run a notebook cell.

---

## What You Will Need

Check that you have these four things before running anything:

- **A terminal.** On Mac, open the Terminal app. On Windows, open Command Prompt or PowerShell.
- **Python 3.9 or later.** Open your terminal and type `python3 --version` to check. If you do not have Python yet, the [Data Access Guide](../../phase-1/01-data-access-guide.md) has a step-by-step setup checklist at the bottom.
- **Your AURIN username and password.** These are the same credentials you use to log in at [data.aurin.org.au](https://data.aurin.org.au). If you have not signed the Domain API access agreement yet, do that first — the Data Access Guide walks you through it.
- **`utils.py` in the same folder as this notebook.** This is a helper file that handles authentication and tracks your API credit usage. If you downloaded this notebook on its own, you will also need to download `utils.py` from the same source and place it in the same folder. If you are working from the full project folder, it is already there.

---

## Step 1: Install the Required Packages

Packages are add-ons that give Python extra abilities. You only need to install them once.

Open your terminal, paste the line below, and press Enter. Wait until it finishes (it takes about 30 seconds).

```
pip install requests pandas python-dotenv
```

- `requests`: sends questions to the API and receives answers back
- `pandas`: organises data into neat tables
- `python-dotenv`: reads your credentials from a private file so you never have to type them into your code

If you see “already satisfied” for some packages, that is fine, they are already installed.

Then run the cell below to confirm everything is in place.

In [ ]:
# Run this cell first.
# If anything is missing, the output will tell you exactly what to install.

missing = []
for package in ['requests', 'pandas', 'dotenv']:
    try:
        __import__(package)
    except ImportError:
        missing.append('python-dotenv' if package == 'dotenv' else package)

if missing:
    print('Some packages are missing. Run this in your terminal:')
    print()
    print(f'  pip install {" ".join(missing)}')
    print()
    print('Then re-run this cell.')
else:
    print('All packages found. You are good to go.')

---

## Step 2: Save Your Credentials

Your AURIN username and password need to be available to your code, but you should never type them directly into a notebook. Notebooks can accidentally get shared or uploaded to GitHub.

Instead, you store them in a plain text file called `.env` (the dot at the start is intentional, there is no `.txt` extension). This file stays on your computer and is not included in your notebook, so it will not be shared if you send the notebook to someone else.

**Where to create it:** in the same folder as this notebook, or any folder above it. A good practice is to keep it in your project's top-level folder so all your notebooks can share the one file.

**How to create it:**

1. Open the folder where you want to keep the file
2. Create a new plain text file called `.env`
   - On **Windows**: open Notepad, then use File → Save As and choose "All Files" as the file type. Name it `.env`.
   - On **Mac**: open TextEdit, go to Format → Make Plain Text, then save as `.env`.
3. Paste these two lines into the file, replacing the example values with your real AURIN credentials:

```
AURIN_USERNAME=your.email@university.edu.au
AURIN_PASSWORD=your_aurin_password
```

4. Save and close the file.

> **If you are using Git:** add `.env` to your `.gitignore` file so it is never accidentally committed and uploaded to GitHub.

---

## Step 3: Connect to the API

Think of this like signing in to a library. You show your card once at the start and then you can borrow books freely for the rest of your visit.

The notebooks use a helper file called `utils.py` that handles two things: reading your credentials from the `.env` file, and counting how many API credits you use. It must be in the same folder as this notebook (see *What You Will Need* above).

Running the cell below will:
1. Read your credentials from the `.env` file
2. Set up a connection to the Domain API through the AURIN proxy
3. Print your account name so you can confirm the right credentials were loaded

You need to do this once every time you open the notebook.

You will see output similar to this, but with your own email address:

```
Connected to: https://domain.api.aurin.org.au
Logged in as: your.email@university.edu.au

Setup looks good. Ready to make your first call.
```

In [ ]:
import os
import sys

# This line makes Python look in the current folder for our helper file (utils.py)
sys.path.insert(0, '.')

# utils.py reads your .env file automatically when imported
from utils import PROXY_BASE, APICallTracker

# The tracker does two things:
#   1. Adds your credentials to every request automatically
#   2. Keeps count of how many credits you have used this session
tracker = APICallTracker()

username = os.getenv('AURIN_USERNAME', 'not found')
print(f'Connected to: {PROXY_BASE}')
print(f'Logged in as: {username}')
print()

if username == 'not found':
    print('Your .env file was not found or is missing AURIN_USERNAME.')
    print('Check that the .env file is in the main domain_api folder.')
else:
    print('Setup looks good. Ready to make your first call.')

---

## Step 4: Make Your First API Call

We are going to ask the Domain API for one quarter (3 months) of market statistics for Carlton, VIC.

Think of it like this: your code sends a question to Domain’s server. The server checks your credentials, looks up the data, and sends the answer back. The whole thing takes about one second.

We are asking for:
- **Suburb:** Carlton, VIC (postcode 3053)
- **Property type:** Houses
- **Time period:** the most recent quarter only

**This will cost 1 credit.** Run the cell below.

In [ ]:
# The URL tells the API which suburb we want.
# Format: /v2/suburbPerformanceStatistics/{state}/{suburb}/{postcode}
url = f'{PROXY_BASE}/v2/suburbPerformanceStatistics/VIC/Carlton/3053'

# These parameters say what kind of data we want
params = {
    'propertyCategory': 'House',  # 'House' or 'Unit'
    'periodSize': 'quarters',      # group data into 3-month periods
    'totalPeriods': 1,             # just the most recent period for this test
}

# Send the request and store the response
response = tracker.get(url, params=params)
tracker.checkpoint('First test call')

# Check the status code: 200 means success
print(f'Status: {response.status_code}')

if response.status_code == 200:
    print(f'Data received: {len(response.text):,} characters')
    print()
    print('It worked. Run the next cell to see the actual numbers.')
elif response.status_code == 401:
    print('Your credentials were not accepted.')
    print('Check AURIN_USERNAME and AURIN_PASSWORD in your .env file.')
elif response.status_code == 403:
    print('Access denied.')
    print('Either the Domain API access agreement is not signed,')
    print('or your monthly credits ran out.')
    print('Log in to data.aurin.org.au and check: Manage Access -> API Access.')
else:
    print(f'Unexpected response: {response.text[:300]}')

---

## Step 5: Look at the Data

The API sent back a chunk of text in a format called JSON. Think of JSON as a structured list of labelled values, a bit like a form that has been filled in.

Python can read JSON like a dictionary. The cell below digs into it and prints the numbers in plain English.

You will see output similar to this, but the figures will reflect the most recent quarter available at the time you run it:

```
Carlton, VIC -- most recent quarter: 2026, month 3

  Median sale price:  $1,305,000
  Properties sold:    18
  Days on market:     not reported
```

"Not reported" means there were not enough sales that quarter for the API to publish a reliable figure.

In [ ]:
# Parse the raw response text into a Python object
data = response.json()

# Dig into the structure to find the most recent quarter
entries = data.get('series', {}).get('seriesInfo', [])

if not entries:
    print('The response came back empty.')
    print('This usually means the suburb name or postcode was wrong.')
    print('Carlton / 3053 should always return data -- re-check the connection cell above.')
else:
    latest = entries[-1]  # last entry = most recent quarter
    year   = latest.get('year')
    month  = latest.get('month')
    values = latest.get('values', {})

    median_price   = values.get('medianSoldPrice')
    number_sold    = values.get('numberSold')
    days_on_market = values.get('daysOnMarket')

    print(f'Carlton, VIC -- most recent quarter: {year}, month {month}')
    print()

    # 'None' means there were not enough sales that quarter for the API to report a number
    if median_price is not None:
        print(f'  Median sale price:  ${median_price:,.0f}')
    else:
        print( '  Median sale price:  not reported this quarter')

    if number_sold is not None:
        print(f'  Properties sold:    {number_sold}')
    else:
        print( '  Properties sold:    not reported')

    if days_on_market is not None:
        print(f'  Days on market:     {days_on_market}')
    else:
        print( '  Days on market:     not reported')

    print()
    print('These numbers come directly from the live Domain API.')

---

## About Credits

Every request you send to the API costs **1 credit**. Your AURIN account gets **1,000 credits per month**. They reset automatically on the 1st of each calendar month, you do not need to do anything.

1,000 sounds like a lot, but it goes quickly if you query without planning first. For example, pulling one year of sold listings across a single busy suburb like South Yarra can take 20–30 credits on its own.

The golden rule: **design your research question before you start querying.** The [Credit Calculator guide](../../phase-1/03-credit-calculator.md) shows you how to estimate your total cost before running anything large.

You can check your remaining credits at any time: log in to [data.aurin.org.au](https://data.aurin.org.au), go to **Manage Access → API Access**, and look at the Domain Access section.

The cell below shows how many credits this session has used so far.

In [ ]:
tracker.summary()

---

## Troubleshooting

**`ModuleNotFoundError: No module named 'utils'`**
The `utils.py` file is not in the same folder as this notebook. Download it from the same source as the notebook and place it in the same folder, then re-run the cell.

**`AssertionError: AURIN_USERNAME not found in .env`**
Your `.env` file is missing or cannot be found. It needs to be in the same folder as this notebook, or in a folder above it. Also double-check the file name: it starts with a dot (`.env`) and has no other extension.

**`Status: 200` but the response came back empty**
The suburb name or postcode was wrong. Carlton / 3053 should always return data. If it does not, re-run the connection cell and try the call again.

**`Status: 401`**
Your credentials were not accepted. Open the `.env` file and check:
- No extra spaces before or after the `=`
- No extra spaces at the beginning or end of the password
- The email address matches exactly what you use to log in to AURIN

**`Status: 403`**
Two possible causes:
1. You have not signed the Domain API access agreement. Log in to data.aurin.org.au → Manage Access → My Agreements.
2. Your monthly credits have run out. Check the dashboard.

**`ModuleNotFoundError: No module named ...`** (for `requests`, `pandas`, or `dotenv`)
A required package is not installed. Run `pip install requests pandas python-dotenv` in your terminal, then re-run the package check cell.

---

## What's Next: Let's Move to Notebook 1

You have connected to the API and pulled real data back. The pattern you just used (build a URL, send a request, check the status, read the result) is the same pattern used in every notebook that follows.

**Notebook 1** covers the listings endpoint: the main data source on the Domain API. It returns individual property records, one per property, with fields like address, price, listing date, and property type. You will learn how to structure a request, apply filters, and retrieve all pages of results.

→ Open `notebook-1-listings-search.ipynb` to continue.